# Placement 1 version 2 — shared questions, shared layer

This notebook implements `docs/placement_1/guide_2.md`. `notebooks/placement_1_v1.ipynb` is the first run and is not changed. Results go to `output/placement_1_v2/` so they do not overwrite that run.

English–Persian is the comparison. Bengali and Urdu are scored in the behavioural screen and kept for the supplement. Questions for SAE, Jacobian Lens, and activation patching are chosen only from that screen. Layer 38 is the shared layer. SAE also trains at layer 20. Patching screens layers 0–41 in both directions.

| Stage | What it writes |
| --- | --- |
| 1 Behavioural screening | `records/baseline_all.json` |
| 2 Question selection | `records/splits.json`, `records/review_sheet.csv` |
| 3 SAE training | `sae/layer_{20,38}/` checkpoints, deleted after stage 4 |
| 4 SAE analysis | `records/sae_features.json`, `records/sae_interventions.json` |
| 5 JLens fitting | `jlens/jacobian_lens.pt`, deleted after stage 6 |
| 6 JLens analysis | `records/jlens_readouts.json`, `records/jlens_interventions.json` |
| 7 Activation patching | `records/patching.json` |
| 8 Method comparison | `records/comparison.json` |
| 9 Report | `records/report_notes.md` |

Each stage reloads its JSON or checkpoint when you run the cell again. Set the matching `FORCE_*` flag to recompute. Set `SELECTED_IDS` after you review `review_sheet.csv` if you want to replace the automatic subset.

The last cell deletes every `.pt` file once the matching analysis stage is `ok`. That leaves the records and figures small enough to download. A failed stage keeps its checkpoints so that stage can resume.


## Setup

On the pod, install the wheel first (`docs/others/runpod_setup.md`). The next cell loads `notebooks/runpod_setup.py`. `MODEL_NAME` stays `/models/gemma-2-9b`.


In [ ]:
from pathlib import Path

try:
    import google.colab  # noqa: F401
    from google.colab import drive
    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")
    print("Drive mounted:", Path("/content/drive/MyDrive").exists())
except ImportError:
    print("Not Colab — skip mount.")


In [ ]:
from pathlib import Path
import os
import runpy


def _resolve_setup_script() -> Path:
    try:
        here = Path.cwd()
    except FileNotFoundError:
        for fallback in ("/workspace/multilingual-mechinterp", "/workspace", "/root", "/"):
            if Path(fallback).is_dir():
                os.chdir(fallback)
                break
        here = Path.cwd()
    candidates = [
        here / "runpod_setup.py",
        here / "notebooks" / "runpod_setup.py",
        here.parent / "notebooks" / "runpod_setup.py",
        Path("/workspace/multilingual-mechinterp/notebooks/runpod_setup.py"),
        Path("/content/drive/MyDrive/multilingual_mech/notebooks/runpod_setup.py"),
    ]
    for path in candidates:
        if path.exists():
            return path
    raise FileNotFoundError(
        "runpod_setup.py not found. On RunPod, start the kernel in the repo or in notebooks/."
    )


_setup = runpy.run_path(str(_resolve_setup_script()))
globals().update({k: _setup[k] for k in _setup["EXPORTS"]})

import csv
import json
import platform
import time
import traceback
from datetime import datetime, timezone

import matplotlib.pyplot as plt
import numpy as np
import torch
import transformers

import multilingual_mechinterp
from multilingual_mechinterp.data import (
    correct_letter,
    load_culture_questions,
    option_token_ids,
    score_mcq_prompt,
    score_options_from_logits,
)
from multilingual_mechinterp.jlens import JacobianLens, fit, from_hf, intervene_jlens, run_jlens
from multilingual_mechinterp.metrics.sae import evaluate_sae
from multilingual_mechinterp.models.hooks import hooked_forward
from multilingual_mechinterp.models.loader import LoadedModel
from multilingual_mechinterp.patching import get_blocks, patch_residuals, run_patching
from multilingual_mechinterp.sae import (
    TiedSAE,
    ablate_features,
    extract_residual_activations,
    steer_features,
    train_tied_sae,
)
from multilingual_mechinterp.sae.features import top_activating_features
from multilingual_mechinterp.utils import ensure_dir, save_json

PLACE = ensure_dir(OUTPUT_DIR / "placement_1_v2")
REC = ensure_dir(PLACE / "records")
FIG = ensure_dir(PLACE / "figures")
SAE_DIR = ensure_dir(PLACE / "sae")
JLENS_DIR = ensure_dir(PLACE / "jlens")

plt.rcParams.update({"figure.figsize": (8, 4), "axes.grid": True, "font.size": 11})
print("ROOT  ", ROOT)
print("OUTPUT", PLACE)
print("MODEL ", MODEL_NAME)


## Run controls

`N_EVAL` is the mechanistic subset (guide target 20–30). It is chosen from behaviour only. `JLENS_FIT_N` is the separate fitting set (guide target 20–30 prompts). Jacobian fitting is the slow stage: about `d_model / JLENS_DIM_BATCH` backward passes per prompt per source layer. Lower `JLENS_FIT_N` or raise `JLENS_DIM_BATCH` if that stage is too long.

Layer 38 is the shared comparison. Layer 29 is off unless `RUN_JLENS_LAYER_29` is True. A second Jacobian fit, used only as a stability check, is off unless `RUN_JLENS_REPEAT_FIT` is True.


In [ ]:
# None scores every question in the culture file (200). An int caps the screen for a smoke test.
SCREEN_LIMIT = None
N_EVAL = 24
N_JLENS_FIT = 20
N_JLENS_HOLDOUT = 4
# None uses the automatic subset. After reviewing review_sheet.csv, pin ids, e.g. [3, 8, 12].
SELECTED_IDS = None

SAE_LAYERS = [20, 38]
SAE_ALPHAS = [3.0e-4, 8.6e-4, 3.0e-3]
SAE_EPOCHS = 3
SAE_BATCH = 256
SAE_RATIO = 2.0
SAE_MAX_TEXTS = 160
SAE_VAL_TEXTS = 40
SAE_MAX_LENGTH = 256
SAE_TOP_K = 10
# Held-out FVU within this band of the best FVU, then lowest mean L0.
SAE_FVU_TOL = 0.05

JLENS_LAYERS = [38, 40]
RUN_JLENS_LAYER_29 = False
JLENS_DIM_BATCH = 32
JLENS_SKIP_FIRST = 16
JLENS_MAX_SEQ = 128
JLENS_TOP_K = 8
RUN_JLENS_REPEAT_FIT = False

PATCH_WINDOW = 1
CONTROL_N = 2
SHARED_LAYER = 38
CASE_N = 5
INTERVENTION_N = 4
STEER_SCALES = [0.25, 0.5]
SAE_STEER_STRENGTHS = [1.0, 2.0]

# Same predicted letter, but treat the pair as B1 when either threshold is crossed.
JSD_B1 = 0.05
P_DIFF_B1 = 0.10

MAX_LENGTH = 512
SEED = 0
LANGS = ("english", "persian", "bengali", "urdu")
FORMAT_TOKENS = {
    "", "a", "b", "c", "d", "answer", ":", ".", ",", "?", "!",
    "<bos>", "<s>", "</s>", "<eos>", "<pad>", "<unk>",
}

RUN_SCREEN = True
RUN_SELECT = True
RUN_SAE = True
RUN_SAE_ANALYSIS = True
RUN_JLENS = True
RUN_JLENS_ANALYSIS = True
RUN_PATCHING = True
RUN_COMPARE = True

FORCE_SCREEN = False
FORCE_SELECT = False
FORCE_SAE = False
FORCE_SAE_ANALYSIS = False
FORCE_JLENS = False
FORCE_JLENS_ANALYSIS = False
FORCE_PATCHING = False
FORCE_COMPARE = False

torch.manual_seed(SEED)
STATUS = {}
STAGE_TIMES = {}
if (REC / "status.json").exists():
    STATUS = json.loads((REC / "status.json").read_text(encoding="utf-8"))
if (REC / "stage_times.json").exists():
    STAGE_TIMES = json.loads((REC / "stage_times.json").read_text(encoding="utf-8"))


In [ ]:
if USE_TINY_OFFLINE:
    raise RuntimeError("Placement 1 needs Gemma 2 9B. USE_TINY_OFFLINE must stay False.")

print("Loading", MODEL_NAME)
model = load_experiment_model(MODEL_NAME)
if not isinstance(model, LoadedModel):
    raise TypeError(f"Expected LoadedModel, got {type(model).__name__}")

param = next(model.model.parameters())
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"
env_record = {
    "created_at": datetime.now(timezone.utc).isoformat(),
    "model_name": model.name,
    "n_layers": model.n_layers,
    "d_model": int(param.shape[-1]),
    "dtype": str(param.dtype),
    "device": str(model.device),
    "gpu": gpu_name,
    "torch": torch.__version__,
    "transformers": transformers.__version__,
    "cuda": torch.version.cuda,
    "platform": platform.platform(),
    "package": getattr(multilingual_mechinterp, "__version__", None),
    "seed": SEED,
    "hook": "blocks.{layer} output, residual stream after the transformer block (resid_post)",
    "token_position": "last non-padding token for scoring, SAE analysis, JLens readouts, and patching",
    "sae_training_positions": "all non-padding tokens in the training prompts",
    "prompt_format": "question, blank line, A.–D. choices, blank line, Answer:",
    "scoring": "softmax over the first token of A B C D; margin is correct logit minus max wrong logit",
    "shared_layer": SHARED_LAYER,
    "sae_layers": SAE_LAYERS,
    "jlens_layers": JLENS_LAYERS,
    "patch_window": PATCH_WINDOW,
}
save_json(env_record, REC / "environment.json")
print(json.dumps({k: env_record[k] for k in ("model_name", "n_layers", "dtype", "gpu", "torch", "transformers")}, indent=2))


## Helpers

Stages reload JSON from `records/` unless the matching `FORCE_*` flag is set. Training activations and the fitted lens are files, so a restarted kernel can continue.


In [ ]:
def dump(name, obj):
    path = save_json(obj, REC / f"{name}.json")
    print("saved", path)
    return path


def load_record(name):
    path = REC / f"{name}.json"
    if not path.exists():
        return None
    return json.loads(path.read_text(encoding="utf-8"))


def savefig(fig, name):
    path = FIG / name
    fig.savefig(path, dpi=160, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    print("saved", path)
    return str(path)


class timed:
    def __init__(self, name):
        self.name = name

    def __enter__(self):
        self.t0 = time.perf_counter()
        if torch.cuda.is_available():
            torch.cuda.reset_peak_memory_stats()
        print(f"--- {self.name} ---")
        return self

    def __exit__(self, exc_type, exc, tb):
        dt = time.perf_counter() - self.t0
        gpu_gb = None
        if torch.cuda.is_available():
            gpu_gb = round(torch.cuda.max_memory_allocated() / 1024**3, 2)
        STAGE_TIMES[self.name] = {"seconds": round(dt, 2), "peak_gpu_gb": gpu_gb}
        save_json(STAGE_TIMES, REC / "stage_times.json")
        label = "failed" if exc_type else "ok"
        print(f"--- {self.name} {label} in {dt:.1f}s  peak_gpu_gb={gpu_gb} ---")
        return False


def write_status(stage, value):
    STATUS[stage] = value
    save_json(STATUS, REC / "status.json")


def empty_cache():
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def prob_vector(score):
    return np.array([float(score["probs"][letter]) for letter in "ABCD"], dtype=float)


def entropy_bits(probs):
    p = np.asarray(probs, dtype=float)
    p = p / p.sum()
    return float(-(p * np.log2(np.clip(p, 1e-12, None))).sum())


def js_divergence(p, q):
    p = np.asarray(p, dtype=float)
    q = np.asarray(q, dtype=float)
    p = p / p.sum()
    q = q / q.sum()
    mid = 0.5 * (p + q)

    def _kl(a, b):
        mask = a > 0
        return float(np.sum(a[mask] * np.log2(a[mask] / np.clip(b[mask], 1e-12, None))))

    return 0.5 * _kl(p, mid) + 0.5 * _kl(q, mid)


def assign_group(english, persian, jsd):
    if english["correct"] and not persian["correct"]:
        return "A1"
    if persian["correct"] and not english["correct"]:
        return "A2"
    if (not english["correct"]) and (not persian["correct"]) and english["pred_letter"] != persian["pred_letter"]:
        return "A3"
    conf = abs(float(english["p_correct"]) - float(persian["p_correct"]))
    if jsd >= JSD_B1 or conf >= P_DIFF_B1:
        return "B1"
    return "B2"


def encode_prompt(prompt):
    tok = model.tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_LENGTH)
    tok = {k: v.to(model.device) for k, v in tok.items()}
    mask = tok.get("attention_mask")
    if mask is None:
        pos = int(tok["input_ids"].shape[-1]) - 1
    else:
        pos = int(mask[0].sum().item()) - 1
    return tok, pos


def n_tokens(prompt):
    tok = model.tokenizer(prompt, truncation=True, max_length=MAX_LENGTH, return_tensors="pt")
    return int(tok["input_ids"].shape[-1])


def last_residual(prompt, layer):
    tok, pos = encode_prompt(prompt)
    with torch.no_grad(), hooked_forward(model.model, layers=[int(layer)]) as cache:
        model.model(**tok)
        acts = cache.get(f"blocks.{int(layer)}")
    return acts[0, pos].detach().float().cpu(), pos


def logits_with_residual(prompt, layer, residual, pos):
    tok, pos = encode_prompt(prompt)
    dtype = next(model.model.parameters()).dtype
    values = residual.detach().to(device=model.device, dtype=dtype).unsqueeze(0)
    positions = torch.tensor([pos], device=model.device)
    with torch.no_grad(), patch_residuals(model.model, {int(layer): values}, positions):
        out = model.model(**tok)
    return out.logits[0, pos].detach().float().cpu()


def slim_score(score):
    probs = {k: float(v) for k, v in score["probs"].items()}
    return {
        "pred_letter": score["pred_letter"],
        "correct": bool(score["correct"]),
        "p_correct": float(score["p_correct"]),
        "margin": float(score["margin"]),
        "probs": probs,
        "entropy": entropy_bits([probs[letter] for letter in "ABCD"]),
    }


def score_prompt(prompt, letter):
    return slim_score(score_mcq_prompt(model, prompt, correct_letter=letter))


def score_logits(logits_row, letter):
    opt_ids = option_token_ids(model.tokenizer)
    return slim_score(score_options_from_logits(logits_row, opt_ids, correct_letter=letter))


def jaccard(left, right):
    a, b = set(left), set(right)
    if not a and not b:
        return 1.0
    return len(a & b) / len(a | b)


def is_format_token(token):
    text = token.strip().lower()
    return text in FORMAT_TOKENS or text.startswith("<")


def spearman(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    if len(x) < 3 or np.std(x) == 0 or np.std(y) == 0:
        return None

    def _rank(values):
        order = np.argsort(values)
        ranks = np.empty(len(values), dtype=float)
        ranks[order] = np.arange(len(values))
        return ranks

    return float(np.corrcoef(_rank(x), _rank(y))[0, 1])


def items_by_id():
    return {int(item.question_number): item for item in load_culture_questions(culture_json_path())}


BASELINE = load_record("baseline_all")
SPLITS = load_record("splits")
LENS = None
LENS_MODEL = None
print(
    "helpers ready",
    "baseline",
    None if BASELINE is None else len(BASELINE),
    "eval",
    None if SPLITS is None else len(SPLITS.get("eval_ids", [])),
)


## Stage 1 — Behavioural screening

Score every question in English, Persian, Bengali, and Urdu with the same prompt and the same A–D scoring. The saved file is the baseline used for every later choice. Groups A1–B2 use English versus Persian only.


In [ ]:
BASELINE = load_record("baseline_all")
if not RUN_SCREEN:
    print("RUN_SCREEN is False")
    write_status("screen", "skipped")
elif BASELINE is not None and not FORCE_SCREEN:
    print(f"loaded baseline_all.json ({len(BASELINE)} questions)")
    write_status("screen", "loaded")
else:
    BASELINE = []
    try:
        with timed("screen"):
            questions = load_culture_questions(culture_json_path(), limit=SCREEN_LIMIT)
            print(f"scoring {len(questions)} questions x {len(LANGS)} languages")
            for index, item in enumerate(questions, start=1):
                letter = correct_letter(item.correct_index)
                languages = {}
                for lang in LANGS:
                    languages[lang] = score_prompt(item.mcq_prompt(lang), letter)
                en = prob_vector(languages["english"])
                jsd = {
                    "english_persian": js_divergence(en, prob_vector(languages["persian"])),
                    "english_bengali": js_divergence(en, prob_vector(languages["bengali"])),
                    "english_urdu": js_divergence(en, prob_vector(languages["urdu"])),
                }
                BASELINE.append(
                    {
                        "id": int(item.question_number),
                        "difficulty": item.difficulty,
                        "correct_letter": letter,
                        "languages": languages,
                        "jsd": {key: float(value) for key, value in jsd.items()},
                        "group": assign_group(
                            languages["english"],
                            languages["persian"],
                            jsd["english_persian"],
                        ),
                    }
                )
                if index % 25 == 0 or index == len(questions):
                    print(f"  {index}/{len(questions)}")
            dump("baseline_all", BASELINE)
        write_status("screen", "ok")
    except Exception:
        write_status("screen", "failed")
        traceback.print_exc()
    empty_cache()

if BASELINE:
    counts = {lang: float(np.mean([row["languages"][lang]["correct"] for row in BASELINE])) for lang in LANGS}
    groups = {name: sum(row["group"] == name for row in BASELINE) for name in ("A1", "A2", "A3", "B1", "B2")}
    print("accuracy", {k: round(v, 3) for k, v in counts.items()})
    print("groups", groups)
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].bar(list(counts), list(counts.values()), color="#4C78A8")
    axes[0].set_ylim(0, 1)
    axes[0].set_ylabel("Accuracy")
    axes[0].set_title(f"Behavioural accuracy (n={len(BASELINE)})")
    jsd_values = [row["jsd"]["english_persian"] for row in BASELINE]
    axes[1].hist(jsd_values, bins=20, color="#F58518")
    axes[1].set_xlabel("JSD English–Persian (bits)")
    axes[1].set_title("Option-distribution divergence")
    savefig(fig, "01_baseline_screen.png")


## Stage 2 — Question selection

The mechanistic subset is a mix of A1, A2, A3, B1, and B2, ranked by English–Persian divergence inside each group. SAE training prompts, Jacobian fitting prompts, and the evaluation questions are disjoint. Nothing in this cell reads SAE, JLens, or patching results.

`review_sheet.csv` is the sheet to read before treating the subset as final. To replace the automatic ids, set `SELECTED_IDS` in the control cell and set `FORCE_SELECT = True`.


In [ ]:
SPLITS = load_record("splits")
if not RUN_SELECT:
    print("RUN_SELECT is False")
    write_status("select", "skipped")
elif BASELINE is None:
    print("baseline_all.json is missing. Run stage 1 first.")
    write_status("select", "blocked")
elif SPLITS is not None and not FORCE_SELECT and SELECTED_IDS is None:
    print(f"loaded splits.json ({len(SPLITS['eval_ids'])} eval questions)")
    write_status("select", "loaded")
else:
    try:
        with timed("select"):
            by_group = {name: [] for name in ("A1", "A2", "A3", "B1", "B2")}
            for row in BASELINE:
                by_group[row["group"]].append(row)
            for name, rows in by_group.items():
                rows.sort(key=lambda row: row["jsd"]["english_persian"], reverse=(name != "B2"))
            group_counts = {name: len(rows) for name, rows in by_group.items()}
            print("available", group_counts)

            if SELECTED_IDS is not None:
                eval_ids = [int(value) for value in SELECTED_IDS]
                missing = [value for value in eval_ids if value not in {row["id"] for row in BASELINE}]
                if missing:
                    raise ValueError(f"SELECTED_IDS not in the screen: {missing}")
            else:
                weight = {"A1": 6, "A2": 6, "A3": 4, "B1": 4, "B2": 4}
                scale = N_EVAL / sum(weight.values())
                quotas = {name: int(round(weight[name] * scale)) for name in weight}
                while sum(quotas.values()) > N_EVAL:
                    donor = max(quotas, key=quotas.get)
                    quotas[donor] -= 1
                while sum(quotas.values()) < N_EVAL:
                    quotas["A1"] += 1
                eval_ids = []
                used = set()
                for name in ("A1", "A2", "A3", "B1", "B2"):
                    take = by_group[name][: min(quotas[name], len(by_group[name]))]
                    for row in take:
                        eval_ids.append(row["id"])
                        used.add(row["id"])
                if len(eval_ids) < N_EVAL:
                    pool = sorted(
                        [row for row in BASELINE if row["id"] not in used and row["group"] in ("A1", "A2", "A3", "B1")],
                        key=lambda row: row["jsd"]["english_persian"],
                        reverse=True,
                    )
                    for row in pool:
                        if len(eval_ids) >= N_EVAL:
                            break
                        eval_ids.append(row["id"])
                eval_ids = eval_ids[:N_EVAL]

            eval_set = set(eval_ids)
            rng = np.random.default_rng(SEED)
            buckets = {}
            for row in BASELINE:
                if row["id"] in eval_set:
                    continue
                buckets.setdefault(row["difficulty"] or "unspecified", []).append(row["id"])
            for key in buckets:
                rng.shuffle(buckets[key])
            mixed = []
            keys = sorted(buckets)
            while any(buckets.values()):
                for key in keys:
                    if buckets[key]:
                        mixed.append(int(buckets[key].pop()))
            need = N_JLENS_FIT + N_JLENS_HOLDOUT
            if len(mixed) <= need:
                raise RuntimeError(
                    f"Only {len(mixed)} questions left after the eval subset; "
                    "lower N_EVAL, N_JLENS_FIT, or N_JLENS_HOLDOUT."
                )
            fit_ids = mixed[:N_JLENS_FIT]
            hold_ids = mixed[N_JLENS_FIT:need]
            sae_ids = mixed[need:]
            n_val = max(8, int(round(0.15 * len(sae_ids))))
            n_val = min(n_val, max(1, len(sae_ids) - 4))
            sae_val = sae_ids[:n_val]
            sae_train = sae_ids[n_val:]
            if len(sae_train) < 4 or len(sae_val) < 1:
                raise RuntimeError(
                    "Not enough leftover questions for a SAE train/validation split. "
                    "Lower N_EVAL or N_JLENS_FIT."
                )
            by_id = {row["id"]: row for row in BASELINE}
            SPLITS = {
                "created_at": datetime.now(timezone.utc).isoformat(),
                "seed": SEED,
                "selection": "manual SELECTED_IDS" if SELECTED_IDS is not None else "behavioural quotas",
                "jsd_b1": JSD_B1,
                "p_diff_b1": P_DIFF_B1,
                "group_counts": {name: sum(row["group"] == name for row in BASELINE) for name in ("A1", "A2", "A3", "B1", "B2")},
                "eval_ids": eval_ids,
                "eval_groups": {str(value): by_id[value]["group"] for value in eval_ids},
                "jlens_fit_ids": fit_ids,
                "jlens_holdout_ids": hold_ids,
                "sae_train_ids": sae_train,
                "sae_val_ids": sae_val,
                "note": (
                    "Eval, Jacobian fitting, Jacobian holdout, SAE train, and SAE validation "
                    "are disjoint. Selection used English–Persian behaviour only."
                ),
            }
            dump("splits", SPLITS)
            catalog = items_by_id()
            review_path = REC / "review_sheet.csv"
            with review_path.open("w", newline="", encoding="utf-8") as handle:
                writer = csv.DictWriter(
                    handle,
                    fieldnames=[
                        "id", "group", "jsd_en_fa", "correct_letter",
                        "en_pred", "fa_pred", "en_correct", "fa_correct",
                        "en_p_correct", "fa_p_correct", "en_question", "fa_question",
                    ],
                )
                writer.writeheader()
                for value in eval_ids:
                    row = by_id[value]
                    item = catalog[value]
                    writer.writerow(
                        {
                            "id": value,
                            "group": row["group"],
                            "jsd_en_fa": round(row["jsd"]["english_persian"], 4),
                            "correct_letter": row["correct_letter"],
                            "en_pred": row["languages"]["english"]["pred_letter"],
                            "fa_pred": row["languages"]["persian"]["pred_letter"],
                            "en_correct": row["languages"]["english"]["correct"],
                            "fa_correct": row["languages"]["persian"]["correct"],
                            "en_p_correct": round(row["languages"]["english"]["p_correct"], 4),
                            "fa_p_correct": round(row["languages"]["persian"]["p_correct"], 4),
                            "en_question": item.text("english"),
                            "fa_question": item.text("persian"),
                        }
                    )
            print("saved", review_path)
            print("eval groups", SPLITS["eval_groups"])
        write_status("select", "ok")
    except Exception:
        write_status("select", "failed")
        traceback.print_exc()


## Stage 3 — SAE training

Train a separate tied SAE at layers 20 and 38 on multilingual prompts that are not in the evaluation set and not in the Jacobian fitting set. The validation split is a different set of those leftover questions. The checkpoint that is kept is chosen on held-out FVU and held-out mean L0, then frozen.

A few validation prompts are rescored with the reconstructed residual so the record shows whether reconstruction changes the letter, `p_correct`, or the margin.


In [ ]:
SAE_RUNS = load_record("sae_training") or []
if not RUN_SAE:
    print("RUN_SAE is False")
    write_status("sae_train", "skipped")
elif SPLITS is None:
    print("splits.json is missing. Run stage 2 first.")
    write_status("sae_train", "blocked")
else:
    try:
        with timed("sae_train"):
            catalog = items_by_id()
            device = "cuda" if torch.cuda.is_available() else "cpu"

            def prompt_list(ids, limit):
                texts = []
                for value in ids:
                    item = catalog[int(value)]
                    for lang in LANGS:
                        texts.append(item.mcq_prompt(lang))
                rng = np.random.default_rng(SEED)
                order = rng.permutation(len(texts))
                return [texts[int(i)] for i in order[:limit]]

            train_texts = prompt_list(SPLITS["sae_train_ids"], SAE_MAX_TEXTS)
            val_texts = prompt_list(SPLITS["sae_val_ids"], SAE_VAL_TEXTS)
            print(f"SAE texts train={len(train_texts)} val={len(val_texts)} device={device}")
            SAE_RUNS = []
            fig, ax = plt.subplots(figsize=(7, 4))
            for layer in SAE_LAYERS:
                layer_dir = ensure_dir(SAE_DIR / f"layer_{layer}")
                metrics_path = layer_dir / "sweep_metrics.json"
                chosen_path = layer_dir / "chosen.pt"
                if chosen_path.exists() and metrics_path.exists() and not FORCE_SAE:
                    rows = json.loads(metrics_path.read_text(encoding="utf-8"))
                    print(f"layer {layer}: loaded {chosen_path.name}")
                else:
                    train_path = layer_dir / "train_acts.pt"
                    val_path = layer_dir / "val_acts.pt"
                    if train_path.exists() and val_path.exists() and not FORCE_SAE:
                        train_acts = torch.load(train_path, map_location="cpu", weights_only=True)
                        val_acts = torch.load(val_path, map_location="cpu", weights_only=True)
                    else:
                        train_acts = extract_residual_activations(
                            model, train_texts, layer=int(layer), max_length=SAE_MAX_LENGTH, batch_size=2,
                        )
                        val_acts = extract_residual_activations(
                            model, val_texts, layer=int(layer), max_length=SAE_MAX_LENGTH, batch_size=2,
                        )
                        torch.save(train_acts, train_path)
                        torch.save(val_acts, val_path)
                    print(f"layer {layer}: train tokens={tuple(train_acts.shape)} val tokens={tuple(val_acts.shape)}")
                    rows = []
                    for alpha in SAE_ALPHAS:
                        trained = train_tied_sae(
                            train_acts,
                            ratio=SAE_RATIO,
                            alpha=float(alpha),
                            batch_size=SAE_BATCH,
                            n_epochs=SAE_EPOCHS,
                            device=device,
                            seed=SEED,
                        )
                        train_metrics = evaluate_sae(trained.sae, train_acts[: min(4000, train_acts.shape[0])])
                        val_metrics = evaluate_sae(trained.sae, val_acts)
                        ckpt = layer_dir / f"tied_sae_alpha_{alpha:.2e}.pt"
                        trained.sae.cpu().save(ckpt)
                        rows.append(
                            {
                                "layer": int(layer),
                                "alpha": float(alpha),
                                "checkpoint": str(ckpt),
                                "n_train_tokens": int(train_acts.shape[0]),
                                "n_val_tokens": int(val_acts.shape[0]),
                                "history": trained.history,
                                "train_fvu": train_metrics["fvu"],
                                "train_r2": train_metrics["r_squared"],
                                "train_mean_l0": train_metrics["mean_l0"],
                                "train_dead_fraction": train_metrics["dead_fraction"],
                                "val_fvu": val_metrics["fvu"],
                                "val_r2": val_metrics["r_squared"],
                                "val_mean_l0": val_metrics["mean_l0"],
                                "val_dead_fraction": val_metrics["dead_fraction"],
                                "n_feats": val_metrics["n_feats"],
                            }
                        )
                        print(
                            f"  L{layer} alpha={alpha:.2e} val FVU={val_metrics['fvu']:.4f} "
                            f"R2={val_metrics['r_squared']:.4f} L0={val_metrics['mean_l0']:.1f} "
                            f"dead={val_metrics['dead_fraction']:.2%}"
                        )
                        del trained
                        empty_cache()
                    best_fvu = min(row["val_fvu"] for row in rows)
                    pool = [row for row in rows if row["val_fvu"] <= best_fvu + SAE_FVU_TOL]
                    chosen = min(pool, key=lambda row: row["val_mean_l0"])
                    TiedSAE.load(chosen["checkpoint"]).save(chosen_path)
                    for row in rows:
                        row["chosen"] = row["checkpoint"] == chosen["checkpoint"]
                    save_json(rows, metrics_path)
                chosen_rows = [row for row in rows if row.get("chosen")]
                chosen = chosen_rows[0] if chosen_rows else min(rows, key=lambda row: row["val_mean_l0"])
                SAE_RUNS.extend(rows)
                ax.plot(
                    [row["val_mean_l0"] for row in rows],
                    [row["val_fvu"] for row in rows],
                    marker="o",
                    label=f"layer {layer}",
                )
            ax.set_xlabel("Held-out mean L0")
            ax.set_ylabel("Held-out FVU")
            ax.set_title("SAE reconstruction versus sparsity")
            ax.legend()
            savefig(fig, "02_sae_pareto.png")
            dump("sae_training", SAE_RUNS)

            # Reconstruction versus the original prediction, on a few validation prompts.
            recon_rows = []
            probe_ids = list(SPLITS["sae_val_ids"])[:3]
            for layer in SAE_LAYERS:
                sae = TiedSAE.load(SAE_DIR / f"layer_{layer}" / "chosen.pt")
                for value in probe_ids:
                    item = catalog[int(value)]
                    letter = correct_letter(item.correct_index)
                    prompt = item.mcq_prompt("english")
                    clean = score_prompt(prompt, letter)
                    residual, pos = last_residual(prompt, layer)
                    reconstructed = sae.predict(residual.unsqueeze(0)).squeeze(0).detach().float().cpu()
                    scored = score_logits(logits_with_residual(prompt, layer, reconstructed, pos), letter)
                    recon_rows.append(
                        {
                            "id": int(value),
                            "layer": int(layer),
                            "clean_pred": clean["pred_letter"],
                            "recon_pred": scored["pred_letter"],
                            "clean_p_correct": clean["p_correct"],
                            "recon_p_correct": scored["p_correct"],
                            "clean_margin": clean["margin"],
                            "recon_margin": scored["margin"],
                            "letter_changed": clean["pred_letter"] != scored["pred_letter"],
                        }
                    )
            dump("sae_reconstruction_effect", recon_rows)
        write_status("sae_train", "ok")
    except Exception:
        write_status("sae_train", "failed")
        traceback.print_exc()
    empty_cache()


## Stage 4 — SAE feature analysis and interventions

Encode the last-token residual of each selected English–Persian pair at layers 20 and 38. Overlap is the Jaccard of the top features. A feature whose decoder direction’s top token is a letter, a BOS marker, or the word “Answer” is flagged as format-like and is not treated as a cultural concept.

Interventions run on a few items. Ablation and steering are compared with the reconstruction baseline, and a near-zero control feature is edited the same way.


In [ ]:
SAE_FEATURES = load_record("sae_features")
if not RUN_SAE_ANALYSIS:
    print("RUN_SAE_ANALYSIS is False")
    write_status("sae_analysis", "skipped")
elif SPLITS is None or not (SAE_DIR / "layer_38" / "chosen.pt").exists():
    print("Need splits.json and a trained layer-38 SAE.")
    write_status("sae_analysis", "blocked")
elif SAE_FEATURES is not None and not FORCE_SAE_ANALYSIS:
    print(f"loaded sae_features.json ({len(SAE_FEATURES)} rows)")
    write_status("sae_analysis", "loaded")
else:
    try:
        with timed("sae_analysis"):
            catalog = items_by_id()
            by_id = {row["id"]: row for row in BASELINE}
            unembed = model.model.get_output_embeddings().weight.detach().float().cpu()
            SAE_FEATURES = []
            interventions = []
            intervened = 0
            seen_groups = set()
            for layer in SAE_LAYERS:
                sae = TiedSAE.load(SAE_DIR / f"layer_{layer}" / "chosen.pt").cpu()
                for value in SPLITS["eval_ids"]:
                    item = catalog[int(value)]
                    letter = correct_letter(item.correct_index)
                    encoded = {}
                    residuals = {}
                    for lang, key in (("english", "en"), ("persian", "fa")):
                        residual, _pos = last_residual(item.mcq_prompt(lang), layer)
                        residuals[key] = residual
                        latent = sae.encode(sae.center(residual.unsqueeze(0))).squeeze(0).detach().cpu()
                        encoded[key] = latent
                    en_top = top_activating_features(encoded["en"], SAE_TOP_K)
                    fa_top = top_activating_features(encoded["fa"], SAE_TOP_K)
                    en_ids = [row["feature_id"] for row in en_top]
                    fa_ids = [row["feature_id"] for row in fa_top]
                    union = list(dict.fromkeys(en_ids + fa_ids))
                    diffs = []
                    for fid in union:
                        diffs.append(
                            {
                                "feature_id": int(fid),
                                "en": float(encoded["en"][fid]),
                                "fa": float(encoded["fa"][fid]),
                                "abs_diff": float(abs(encoded["en"][fid] - encoded["fa"][fid])),
                            }
                        )
                    diffs.sort(key=lambda row: row["abs_diff"], reverse=True)
                    focus = diffs[0]["feature_id"] if diffs else None
                    top_token = None
                    format_like = None
                    if focus is not None:
                        direction = sae.encoder[focus].detach().float().cpu()
                        token_id = int(torch.matmul(unembed, direction).argmax().item())
                        top_token = model.tokenizer.decode([token_id])
                        format_like = is_format_token(top_token)
                    group = by_id[int(value)]["group"]
                    SAE_FEATURES.append(
                        {
                            "id": int(value),
                            "group": group,
                            "layer": int(layer),
                            "jaccard": jaccard(en_ids, fa_ids),
                            "en_top": en_ids,
                            "fa_top": fa_ids,
                            "largest_diff_feature": focus,
                            "largest_abs_diff": None if not diffs else diffs[0]["abs_diff"],
                            "feature_top_token": top_token,
                            "format_like": format_like,
                        }
                    )
                    if layer != SHARED_LAYER or intervened >= INTERVENTION_N or group in seen_groups:
                        continue
                    if focus is None:
                        continue
                    seen_groups.add(group)
                    intervened += 1
                    prompt = item.mcq_prompt("persian")
                    residual = residuals["fa"]
                    latent = encoded["fa"]
                    busy = set(en_ids + fa_ids)
                    control_id = min(
                        [fid for fid in range(latent.numel()) if fid not in busy],
                        key=lambda fid: abs(float(latent[fid])),
                    )
                    clean = score_prompt(prompt, letter)
                    recon = sae.predict(residual.unsqueeze(0)).squeeze(0).detach().float().cpu()
                    _tok, pos = encode_prompt(prompt)
                    recon_score = score_logits(logits_with_residual(prompt, layer, recon, pos), letter)
                    variants = {"reconstruction": recon_score}
                    for name, edited in (
                        ("ablate_focus", ablate_features(residual, sae, [focus])),
                        ("ablate_control", ablate_features(residual, sae, [int(control_id)])),
                    ):
                        variants[name] = score_logits(logits_with_residual(prompt, layer, edited, pos), letter)
                    for strength in SAE_STEER_STRENGTHS:
                        steered = steer_features(residual, sae, [focus], strength=float(strength))
                        variants[f"steer_focus_{strength}"] = score_logits(
                            logits_with_residual(prompt, layer, steered, pos), letter
                        )
                    interventions.append(
                        {
                            "id": int(value),
                            "group": group,
                            "layer": int(layer),
                            "focus_feature": int(focus),
                            "control_feature": int(control_id),
                            "format_like": format_like,
                            "feature_top_token": top_token,
                            "clean": clean,
                            "variants": variants,
                            "versus_reconstruction": {
                                name: {
                                    "delta_p_correct": float(score["p_correct"] - recon_score["p_correct"]),
                                    "delta_margin": float(score["margin"] - recon_score["margin"]),
                                    "pred_changed": score["pred_letter"] != recon_score["pred_letter"],
                                }
                                for name, score in variants.items()
                                if name != "reconstruction"
                            },
                        }
                    )
            dump("sae_features", SAE_FEATURES)
            dump("sae_interventions", interventions)
            layer_rows = [row for row in SAE_FEATURES if row["layer"] == SHARED_LAYER]
            if layer_rows:
                fig, ax = plt.subplots(figsize=(7, 4))
                labels = []
                means = []
                for name in ("A1", "A2", "A3", "B1", "B2"):
                    vals = [row["jaccard"] for row in layer_rows if row["group"] == name]
                    if vals:
                        labels.append(name)
                        means.append(float(np.mean(vals)))
                ax.bar(labels, means, color="#54A24B")
                ax.set_ylim(0, 1)
                ax.set_ylabel("Top-feature Jaccard")
                ax.set_title(f"English–Persian SAE overlap at layer {SHARED_LAYER}")
                savefig(fig, "03_sae_jaccard_by_group.png")
        write_status("sae_analysis", "ok")
    except Exception:
        write_status("sae_analysis", "failed")
        traceback.print_exc()
    empty_cache()


## Stage 5 — Jacobian Lens fitting

Fit on the held-out fitting prompts, alternating English and Persian, never on the evaluation questions. Source layers are 38 and 40. The fit checkpoint is resumed if the cell is interrupted. Holdout prompts are scored after the fit and are not part of the average Jacobian.

`RUN_JLENS_REPEAT_FIT` fits a second lens on half of the fitting prompts and records token overlap on the holdout. Leave it False on the first pass.


In [ ]:
def fit_prompts(ids):
    catalog = items_by_id()
    prompts = []
    for index, value in enumerate(ids):
        lang = "english" if index % 2 == 0 else "persian"
        prompts.append(catalog[int(value)].mcq_prompt(lang))
    return prompts


LENS = None
LENS_MODEL = None
if not RUN_JLENS:
    print("RUN_JLENS is False")
    write_status("jlens_fit", "skipped")
elif SPLITS is None:
    print("splits.json is missing. Run stage 2 first.")
    write_status("jlens_fit", "blocked")
else:
    try:
        with timed("jlens_fit"):
            layers = list(JLENS_LAYERS)
            if RUN_JLENS_LAYER_29:
                layers = sorted(set(layers + [29]))
            lens_path = JLENS_DIR / "jacobian_lens.pt"
            LENS_MODEL = from_hf(model.model, model.tokenizer, force_bos=False)
            if lens_path.exists() and not FORCE_JLENS:
                LENS = JacobianLens.load(str(lens_path))
                print(f"loaded {lens_path} n_prompts={LENS.n_prompts} layers={LENS.source_layers}")
            else:
                prompts = fit_prompts(SPLITS["jlens_fit_ids"])
                print(f"fitting {len(prompts)} prompts at layers {layers}")
                LENS = fit(
                    LENS_MODEL,
                    prompts,
                    source_layers=layers,
                    dim_batch=JLENS_DIM_BATCH,
                    max_seq_len=JLENS_MAX_SEQ,
                    skip_first=JLENS_SKIP_FIRST,
                    checkpoint_path=str(JLENS_DIR / "fit_checkpoint.pt"),
                    checkpoint_every=1,
                    resume=not FORCE_JLENS,
                )
                LENS.save(str(lens_path))
                print(f"saved {lens_path} n_prompts={LENS.n_prompts}")
            holdout_rows = []
            for prompt in fit_prompts(SPLITS["jlens_holdout_ids"]):
                readout = run_jlens(
                    LENS_MODEL,
                    prompt,
                    layers=list(LENS.source_layers),
                    top_k=JLENS_TOP_K,
                    lens=LENS,
                    use_jacobian=True,
                    max_seq_len=MAX_LENGTH,
                )
                holdout_rows.append(
                    {
                        "layers": {
                            str(layer): [row["token"] for row in tokens]
                            for layer, tokens in readout.top_tokens.items()
                        }
                    }
                )
            stability = {"holdout_readouts": holdout_rows, "repeat_fit": None}
            if RUN_JLENS_REPEAT_FIT:
                half = fit_prompts(SPLITS["jlens_fit_ids"][: max(2, len(SPLITS["jlens_fit_ids"]) // 2)])
                lens_half = fit(
                    LENS_MODEL,
                    half,
                    source_layers=list(LENS.source_layers),
                    dim_batch=JLENS_DIM_BATCH,
                    max_seq_len=JLENS_MAX_SEQ,
                    skip_first=JLENS_SKIP_FIRST,
                    checkpoint_path=str(JLENS_DIR / "fit_half_checkpoint.pt"),
                    resume=True,
                )
                overlaps = []
                for prompt in fit_prompts(SPLITS["jlens_holdout_ids"]):
                    full = run_jlens(LENS_MODEL, prompt, layers=[SHARED_LAYER], top_k=JLENS_TOP_K, lens=LENS)
                    part = run_jlens(LENS_MODEL, prompt, layers=[SHARED_LAYER], top_k=JLENS_TOP_K, lens=lens_half)
                    full_toks = [row["token"].strip() for row in full.top_tokens.get(SHARED_LAYER, [])]
                    part_toks = [row["token"].strip() for row in part.top_tokens.get(SHARED_LAYER, [])]
                    overlaps.append(jaccard(full_toks, part_toks))
                stability["repeat_fit"] = {
                    "n_prompts": len(half),
                    "holdout_token_jaccard_mean": None if not overlaps else float(np.mean(overlaps)),
                }
            dump("jlens_stability", stability)
        write_status("jlens_fit", "ok")
    except Exception:
        write_status("jlens_fit", "failed")
        traceback.print_exc()
    empty_cache()


## Stage 6 — Jacobian Lens readouts and interventions

Readouts are top tokens at the last position. Overlap uses the token string after stripping. A separate count ignores answer letters and prompt-format tokens. Interventions add or remove a content-token direction at layer 38. The letter `A` is the control direction. Swapping adds the Persian content direction and subtracts the English one. These edits show whether the residual move changes the answer; they do not by themselves name a concept.


In [ ]:
JLENS_ROWS = load_record("jlens_readouts")
if not RUN_JLENS_ANALYSIS:
    print("RUN_JLENS_ANALYSIS is False")
    write_status("jlens_analysis", "skipped")
elif SPLITS is None or not (JLENS_DIR / "jacobian_lens.pt").exists():
    print("Need splits.json and jlens/jacobian_lens.pt.")
    write_status("jlens_analysis", "blocked")
elif JLENS_ROWS is not None and not FORCE_JLENS_ANALYSIS:
    print(f"loaded jlens_readouts.json ({len(JLENS_ROWS)} rows)")
    write_status("jlens_analysis", "loaded")
else:
    try:
        with timed("jlens_analysis"):
            if LENS is None:
                LENS = JacobianLens.load(str(JLENS_DIR / "jacobian_lens.pt"))
            if LENS_MODEL is None:
                LENS_MODEL = from_hf(model.model, model.tokenizer, force_bos=False)
            catalog = items_by_id()
            by_id = {row["id"]: row for row in BASELINE}
            layers = [layer for layer in LENS.source_layers if layer in set(JLENS_LAYERS) or layer == 29]
            JLENS_ROWS = []
            interventions = []
            seen_groups = set()
            weight = model.model.get_output_embeddings().weight.detach().float().cpu()

            def content_token(tokens):
                for row in tokens:
                    if not is_format_token(row["token"]):
                        return row
                return None

            def unit_direction(token_id):
                vec = weight[int(token_id)].detach().float().cpu()
                return vec / vec.norm().clamp_min(1e-8)

            for value in SPLITS["eval_ids"]:
                item = catalog[int(value)]
                letter = correct_letter(item.correct_index)
                group = by_id[int(value)]["group"]
                readouts = {}
                for lang, key in (("english", "en"), ("persian", "fa")):
                    result = run_jlens(
                        LENS_MODEL,
                        item.mcq_prompt(lang),
                        layers=layers,
                        top_k=JLENS_TOP_K,
                        lens=LENS,
                        use_jacobian=True,
                        max_seq_len=MAX_LENGTH,
                    )
                    readouts[key] = {
                        str(layer): result.top_tokens.get(layer, [])
                        for layer in layers
                    }
                overlap = {}
                content_overlap = {}
                for layer in layers:
                    en_toks = [row["token"].strip() for row in readouts["en"].get(str(layer), [])]
                    fa_toks = [row["token"].strip() for row in readouts["fa"].get(str(layer), [])]
                    overlap[str(layer)] = jaccard(en_toks, fa_toks)
                    content_overlap[str(layer)] = jaccard(
                        [tok for tok in en_toks if not is_format_token(tok)],
                        [tok for tok in fa_toks if not is_format_token(tok)],
                    )
                JLENS_ROWS.append(
                    {
                        "id": int(value),
                        "group": group,
                        "en_pred": by_id[int(value)]["languages"]["english"]["pred_letter"],
                        "fa_pred": by_id[int(value)]["languages"]["persian"]["pred_letter"],
                        "token_jaccard": overlap,
                        "content_token_jaccard": content_overlap,
                        "readouts": readouts,
                    }
                )
                if group in seen_groups or len(interventions) >= INTERVENTION_N:
                    continue
                en_content = content_token(readouts["en"].get(str(SHARED_LAYER), []))
                fa_content = content_token(readouts["fa"].get(str(SHARED_LAYER), []))
                if en_content is None or fa_content is None:
                    continue
                seen_groups.add(group)
                prompt = item.mcq_prompt("persian")
                residual, pos = last_residual(prompt, SHARED_LAYER)
                clean = score_prompt(prompt, letter)
                en_dir = unit_direction(en_content["token_id"])
                fa_dir = unit_direction(fa_content["token_id"])
                control_id = option_token_ids(model.tokenizer)["A"]
                control_dir = unit_direction(control_id)
                scale = float(residual.norm().item())
                edits = {}
                for factor in STEER_SCALES:
                    edits[f"strengthen_fa_{factor}"] = intervene_jlens(residual, direction=fa_dir, scale=factor * scale)
                    edits[f"suppress_fa_{factor}"] = intervene_jlens(residual, direction=fa_dir, scale=-factor * scale)
                    edits[f"swap_en_fa_{factor}"] = intervene_jlens(
                        residual, direction=fa_dir - en_dir, scale=factor * scale
                    )
                    edits[f"control_A_{factor}"] = intervene_jlens(residual, direction=control_dir, scale=factor * scale)
                scored = {
                    name: score_logits(logits_with_residual(prompt, SHARED_LAYER, edited, pos), letter)
                    for name, edited in edits.items()
                }
                interventions.append(
                    {
                        "id": int(value),
                        "group": group,
                        "layer": SHARED_LAYER,
                        "en_content_token": en_content["token"],
                        "fa_content_token": fa_content["token"],
                        "clean": clean,
                        "edits": {
                            name: {
                                "pred_letter": score["pred_letter"],
                                "p_correct": score["p_correct"],
                                "margin": score["margin"],
                                "delta_p_correct": float(score["p_correct"] - clean["p_correct"]),
                                "delta_margin": float(score["margin"] - clean["margin"]),
                                "pred_changed": score["pred_letter"] != clean["pred_letter"],
                            }
                            for name, score in scored.items()
                        },
                        "note": "Residual addition of an output-embedding direction. A prediction change is a causal effect of that edit, not identification of a concept.",
                    }
                )
            dump("jlens_readouts", JLENS_ROWS)
            dump("jlens_interventions", interventions)
            if JLENS_ROWS and str(SHARED_LAYER) in JLENS_ROWS[0]["content_token_jaccard"]:
                fig, ax = plt.subplots(figsize=(7, 4))
                labels, means = [], []
                for name in ("A1", "A2", "A3", "B1", "B2"):
                    vals = [
                        row["content_token_jaccard"][str(SHARED_LAYER)]
                        for row in JLENS_ROWS
                        if row["group"] == name
                    ]
                    if vals:
                        labels.append(name)
                        means.append(float(np.mean(vals)))
                ax.bar(labels, means, color="#E45756")
                ax.set_ylim(0, 1)
                ax.set_ylabel("Content-token Jaccard")
                ax.set_title(f"JLens content-token overlap at layer {SHARED_LAYER}")
                savefig(fig, "04_jlens_overlap_by_group.png")
        write_status("jlens_analysis", "ok")
    except Exception:
        write_status("jlens_analysis", "failed")
        traceback.print_exc()
    empty_cache()


## Stage 7 — Bidirectional activation patching

Every selected pair is patched one layer at a time from English into Persian and from Persian into English. The metric is the change in the correct-versus-wrong margin. At the layer with the largest absolute change, one more forward records the new letter, `Δp(correct)`, whether the source letter transferred, and whether the replacement matches the source residual.

Two same-language controls replace an English residual with itself. Two unrelated controls paste a different question’s English residual into the Persian prompt. A large matched effect with a small control effect is the comparison that matters. The patch still does not name a semantic concept.


In [ ]:
PATCH_ROWS = load_record("patching")
if not RUN_PATCHING:
    print("RUN_PATCHING is False")
    write_status("patching", "skipped")
elif SPLITS is None or BASELINE is None:
    print("Need baseline_all.json and splits.json.")
    write_status("patching", "blocked")
elif PATCH_ROWS is not None and not FORCE_PATCHING:
    print(f"loaded patching.json ({len(PATCH_ROWS)} rows)")
    write_status("patching", "loaded")
else:
    try:
        with timed("patching"):
            catalog = items_by_id()
            by_id = {row["id"]: row for row in BASELINE}
            blocks = get_blocks(model.model)
            dtype = next(model.model.parameters()).dtype
            eval_ids = [int(value) for value in SPLITS["eval_ids"]]

            def patch_row(source_prompt, target_prompt, source_pred, target_score, letter, meta):
                result = run_patching(
                    model,
                    source_prompt=source_prompt,
                    target_prompt=target_prompt,
                    answer=letter,
                    window=PATCH_WINDOW,
                    metric="margin",
                )
                layers = [int(layer) for layer in result.layers]
                deltas = [float(result.scores[layer]) for layer in layers]
                pos_i = max(range(len(deltas)), key=lambda i: deltas[i])
                neg_i = min(range(len(deltas)), key=lambda i: deltas[i])
                abs_i = max(range(len(deltas)), key=lambda i: abs(deltas[i]))
                extreme = layers[abs_i]
                source_vec, _src_pos = last_residual(source_prompt, extreme)
                tok, pos = encode_prompt(target_prompt)
                values = source_vec.to(device=model.device, dtype=dtype).unsqueeze(0)
                positions = torch.tensor([pos], device=model.device)
                seen = {}

                def _checker(_module, _inp, out, _pos=pos):
                    hidden = out[0] if isinstance(out, tuple) else out
                    seen["residual"] = hidden[0, _pos].detach().float().cpu()

                with torch.no_grad(), patch_residuals(model.model, {extreme: values}, positions):
                    handle = blocks[extreme].register_forward_hook(_checker)
                    try:
                        out = model.model(**tok)
                    finally:
                        handle.remove()
                patched = score_logits(out.logits[0, pos].detach().float().cpu(), letter)
                if "residual" not in seen:
                    max_abs_diff = float("inf")
                else:
                    roundtrip = source_vec.to(dtype=dtype).float()
                    max_abs_diff = float((seen["residual"] - roundtrip).abs().max().item())
                layer_38 = None
                if SHARED_LAYER in result.scores:
                    layer_38 = float(result.scores[SHARED_LAYER])
                target_correct = bool(target_score["correct"])
                return {
                    **meta,
                    "correct_letter": letter,
                    "source_pred": source_pred,
                    "target_pred": target_score["pred_letter"],
                    "target_correct": target_correct,
                    "layers": layers,
                    "delta_margin": deltas,
                    "baseline_margin": float(result.baseline_score),
                    "baseline_p_correct": float(target_score["p_correct"]),
                    "max_pos_layer": layers[pos_i],
                    "max_pos_delta": deltas[pos_i],
                    "max_neg_layer": layers[neg_i],
                    "max_neg_delta": deltas[neg_i],
                    "extreme_layer": extreme,
                    "extreme_delta": deltas[abs_i],
                    "layer_38_delta": layer_38,
                    "patched_pred": patched["pred_letter"],
                    "patched_p_correct": patched["p_correct"],
                    "patched_margin": patched["margin"],
                    "delta_p_correct": float(patched["p_correct"] - target_score["p_correct"]),
                    "letter_changed": patched["pred_letter"] != target_score["pred_letter"],
                    "source_transferred": patched["pred_letter"] == source_pred,
                    "recovered": (not target_correct) and patched["correct"],
                    "worsened": target_correct and (not patched["correct"]),
                    "replacement_max_abs_diff": max_abs_diff,
                    "replacement_ok": max_abs_diff < 0.05,
                    "source_tokens": n_tokens(source_prompt),
                    "target_tokens": n_tokens(target_prompt),
                }

            PATCH_ROWS = []
            for value in eval_ids:
                item = catalog[value]
                row = by_id[value]
                letter = row["correct_letter"]
                en_prompt = item.mcq_prompt("english")
                fa_prompt = item.mcq_prompt("persian")
                en_score = row["languages"]["english"]
                fa_score = row["languages"]["persian"]
                for direction, source, target, source_pred, target_score in (
                    ("en_to_fa", en_prompt, fa_prompt, en_score["pred_letter"], fa_score),
                    ("fa_to_en", fa_prompt, en_prompt, fa_score["pred_letter"], en_score),
                ):
                    patched = patch_row(
                        source,
                        target,
                        source_pred,
                        target_score,
                        letter,
                        {"id": value, "group": row["group"], "direction": direction, "control": "matched"},
                    )
                    PATCH_ROWS.append(patched)
                    print(
                        f"Q{value} {direction} L{patched['extreme_layer']} "
                        f"Δmargin={patched['extreme_delta']:.3f} "
                        f"{patched['target_pred']}→{patched['patched_pred']}"
                    )
            for value in eval_ids[:CONTROL_N]:
                item = catalog[value]
                row = by_id[value]
                en_prompt = item.mcq_prompt("english")
                en_score = row["languages"]["english"]
                PATCH_ROWS.append(
                    patch_row(
                        en_prompt,
                        en_prompt,
                        en_score["pred_letter"],
                        en_score,
                        row["correct_letter"],
                        {"id": value, "group": row["group"], "direction": "en_to_en", "control": "same_language"},
                    )
                )
            for index, value in enumerate(eval_ids[:CONTROL_N]):
                other = eval_ids[(index + 1) % len(eval_ids)]
                row = by_id[value]
                source_row = by_id[other]
                PATCH_ROWS.append(
                    patch_row(
                        catalog[other].mcq_prompt("english"),
                        catalog[value].mcq_prompt("persian"),
                        source_row["languages"]["english"]["pred_letter"],
                        row["languages"]["persian"],
                        row["correct_letter"],
                        {
                            "id": value,
                            "source_id": other,
                            "group": row["group"],
                            "direction": "unrelated_en_to_fa",
                            "control": "unrelated",
                        },
                    )
                )
            dump("patching", PATCH_ROWS)
            fig, ax = plt.subplots(figsize=(9, 4))
            for direction, color in (("en_to_fa", "#4C78A8"), ("fa_to_en", "#F58518")):
                matched = [row for row in PATCH_ROWS if row["direction"] == direction]
                if not matched:
                    continue
                layers = matched[0]["layers"]
                mean = np.mean([row["delta_margin"] for row in matched], axis=0)
                ax.plot(layers, mean, label=direction, color=color)
            ax.axhline(0, color="black", linewidth=0.8)
            ax.axvline(SHARED_LAYER, color="gray", linestyle="--", linewidth=0.8)
            ax.set_xlabel("Layer replaced")
            ax.set_ylabel("Mean Δ margin")
            ax.set_title("Bidirectional single-layer patch")
            ax.legend()
            savefig(fig, "05_patch_mean_delta_margin.png")
        write_status("patching", "ok")
    except Exception:
        write_status("patching", "failed")
        traceback.print_exc()
    empty_cache()


## Stage 8 — Compare the three methods at layer 38

Each selected question gets one row: SAE feature overlap, JLens content-token overlap, and the patch effect in both directions, all at layer 38. Rank correlations ask whether the questions that look different to one method also look different to the others. Agreement of a layer is not agreement on a concept.

Five case-study ids are saved for the report. They prefer one question from each behavioural group, then the largest absolute English→Persian patch effect at layer 38.


In [ ]:
COMPARISON = load_record("comparison")
if not RUN_COMPARE:
    print("RUN_COMPARE is False")
    write_status("compare", "skipped")
elif SPLITS is None or BASELINE is None:
    print("Need the behavioural split before comparison.")
    write_status("compare", "blocked")
elif COMPARISON is not None and not FORCE_COMPARE:
    print(f"loaded comparison.json ({len(COMPARISON['rows'])} rows)")
    write_status("compare", "loaded")
else:
    try:
        with timed("compare"):
            features = load_record("sae_features") or []
            readouts = load_record("jlens_readouts") or []
            patches = load_record("patching") or []
            by_id = {row["id"]: row for row in BASELINE}
            feat_38 = {(row["id"], row["layer"]): row for row in features}
            lens_by = {row["id"]: row for row in readouts}
            patch_by = {(row["id"], row["direction"]): row for row in patches if row.get("control") == "matched"}
            rows = []
            for value in SPLITS["eval_ids"]:
                value = int(value)
                base = by_id[value]
                feat = feat_38.get((value, SHARED_LAYER))
                lens = lens_by.get(value)
                en_fa = patch_by.get((value, "en_to_fa"))
                fa_en = patch_by.get((value, "fa_to_en"))
                rows.append(
                    {
                        "id": value,
                        "group": base["group"],
                        "en_pred": base["languages"]["english"]["pred_letter"],
                        "fa_pred": base["languages"]["persian"]["pred_letter"],
                        "en_correct": base["languages"]["english"]["correct"],
                        "fa_correct": base["languages"]["persian"]["correct"],
                        "jsd": base["jsd"]["english_persian"],
                        "sae_jaccard_l20": None if (value, 20) not in feat_38 else feat_38[(value, 20)]["jaccard"],
                        "sae_jaccard_l38": None if feat is None else feat["jaccard"],
                        "sae_format_like": None if feat is None else feat["format_like"],
                        "jlens_content_jaccard_l38": None
                        if lens is None
                        else lens["content_token_jaccard"].get(str(SHARED_LAYER)),
                        "patch_en_fa_l38": None if en_fa is None else en_fa["layer_38_delta"],
                        "patch_fa_en_l38": None if fa_en is None else fa_en["layer_38_delta"],
                        "patch_en_fa_extreme_layer": None if en_fa is None else en_fa["extreme_layer"],
                        "source_transferred_en_fa": None if en_fa is None else en_fa["source_transferred"],
                        "recovered_en_fa": None if en_fa is None else en_fa["recovered"],
                        "worsened_en_fa": None if en_fa is None else en_fa["worsened"],
                    }
                )
            usable = [
                row for row in rows
                if row["sae_jaccard_l38"] is not None
                and row["jlens_content_jaccard_l38"] is not None
                and row["patch_en_fa_l38"] is not None
            ]
            correlations = {
                "sae_diff_vs_patch": spearman(
                    [1.0 - row["sae_jaccard_l38"] for row in usable],
                    [abs(row["patch_en_fa_l38"]) for row in usable],
                ),
                "jlens_diff_vs_patch": spearman(
                    [1.0 - row["jlens_content_jaccard_l38"] for row in usable],
                    [abs(row["patch_en_fa_l38"]) for row in usable],
                ),
                "sae_diff_vs_jlens_diff": spearman(
                    [1.0 - row["sae_jaccard_l38"] for row in usable],
                    [1.0 - row["jlens_content_jaccard_l38"] for row in usable],
                ),
                "n": len(usable),
            }
            case_ids = []
            ranked = sorted(
                [row for row in rows if row["patch_en_fa_l38"] is not None],
                key=lambda row: abs(row["patch_en_fa_l38"]),
                reverse=True,
            )
            for name in ("A1", "A2", "A3", "B1", "B2"):
                for row in ranked:
                    if row["group"] == name and row["id"] not in case_ids:
                        case_ids.append(row["id"])
                        break
            for row in ranked:
                if len(case_ids) >= CASE_N:
                    break
                if row["id"] not in case_ids:
                    case_ids.append(row["id"])
            COMPARISON = {
                "shared_layer": SHARED_LAYER,
                "rows": rows,
                "rank_correlations": correlations,
                "case_ids": case_ids[:CASE_N],
                "caveat": (
                    "The eval subset is enriched for behavioural disagreement and is not a "
                    "random sample of the 200 questions. A shared layer does not mean the "
                    "methods found the same concept."
                ),
            }
            dump("comparison", COMPARISON)
            print("rank correlations", correlations)
            print("case ids", COMPARISON["case_ids"])
            present = [row for row in rows if row["patch_en_fa_l38"] is not None]
            if present:
                fig, ax = plt.subplots(figsize=(8, 4))
                xs = np.arange(len(present))
                ax.bar(xs - 0.2, [abs(row["patch_en_fa_l38"]) for row in present], width=0.4, label="|Δmargin| EN→FA")
                if any(row["sae_jaccard_l38"] is not None for row in present):
                    ax.plot(xs, [0 if row["sae_jaccard_l38"] is None else 1 - row["sae_jaccard_l38"] for row in present], marker="o", label="1 − SAE Jaccard")
                ax.set_xticks(xs)
                ax.set_xticklabels([f"{row['id']}:{row['group']}" for row in present], rotation=90, fontsize=8)
                ax.set_title(f"Layer {SHARED_LAYER}: patch effect and SAE difference")
                ax.legend()
                savefig(fig, "06_layer38_comparison.png")
        write_status("compare", "ok")
    except Exception:
        write_status("compare", "failed")
        traceback.print_exc()


## Stage 9 — Report notes

This cell only reads the saved records. It states what was measured, the six-way comparison, and five case studies. Copy from `records/report_notes.md` into the write-up. The disagreement-enriched subset is not a claim about all 200 questions.


In [ ]:
def _get(record, *keys, default="not run"):
    cur = record
    for key in keys:
        if not isinstance(cur, dict) or key not in cur:
            return default
        cur = cur[key]
    return cur


baseline = load_record("baseline_all") or []
splits = load_record("splits") or {}
sae_rows = load_record("sae_training") or []
recon = load_record("sae_reconstruction_effect") or []
features = load_record("sae_features") or []
sae_edits = load_record("sae_interventions") or []
stability = load_record("jlens_stability") or {}
readouts = load_record("jlens_readouts") or []
lens_edits = load_record("jlens_interventions") or []
patches = load_record("patching") or []
comparison = load_record("comparison") or {}
env = load_record("environment") or {}
by_id = {row["id"]: row for row in baseline}

lines = []
lines.append("# Placement 1 version 2 notes")
lines.append("")
lines.append("## Setup")
lines.append("")
lines.append(f"- Model: {env.get('model_name', 'unknown')} ({env.get('dtype')}, {env.get('gpu')})")
lines.append(f"- Torch {env.get('torch')}, Transformers {env.get('transformers')}, CUDA {env.get('cuda')}")
lines.append(f"- Hook: {env.get('hook')}")
lines.append(f"- Token position: {env.get('token_position')}")
lines.append(f"- Package {env.get('package')}, seed {env.get('seed')}")
lines.append(f"- Runtime: {json.dumps(STAGE_TIMES)}")
lines.append("")
lines.append("## Behaviour")
lines.append("")
if baseline:
    for lang in LANGS:
        acc = float(np.mean([row["languages"][lang]["correct"] for row in baseline]))
        lines.append(f"- {lang} accuracy: {acc:.3f} on {len(baseline)} questions")
    counts = {name: sum(row["group"] == name for row in baseline) for name in ("A1", "A2", "A3", "B1", "B2")}
    lines.append(f"- English–Persian groups: {counts}")
    jsd_values = [row["jsd"]["english_persian"] for row in baseline]
    lines.append(
        f"- English–Persian JSD mean {float(np.mean(jsd_values)):.3f}, "
        f"median {float(np.median(jsd_values)):.3f}"
    )
lines.append(
    f"- Mechanistic subset ({len(splits.get('eval_ids', []))} questions) was chosen from behaviour only. "
    "It is enriched for disagreement and is not representative of all 200 questions."
)
lines.append(f"- Eval ids: {splits.get('eval_ids')}")
lines.append("")
lines.append("## SAE")
lines.append("")
chosen = [row for row in sae_rows if row.get("chosen")]
if not chosen and sae_rows:
    chosen = sae_rows
for row in chosen:
    lines.append(
        f"- Layer {row['layer']} alpha {row['alpha']:.2e}: held-out FVU {row['val_fvu']:.3f}, "
        f"R² {row['val_r2']:.3f}, mean L0 {row['val_mean_l0']:.1f}, "
        f"dead fraction {row['val_dead_fraction']:.2%}, train tokens {row['n_train_tokens']}."
    )
if recon:
    changed = sum(bool(row["letter_changed"]) for row in recon)
    lines.append(f"- Reconstruction changed the English letter on {changed} of {len(recon)} validation probes.")
feat_38 = [row for row in features if row["layer"] == SHARED_LAYER]
if feat_38:
    lines.append(
        f"- Layer {SHARED_LAYER} mean top-feature Jaccard: {float(np.mean([row['jaccard'] for row in feat_38])):.3f}."
    )
    flagged = sum(bool(row["format_like"]) for row in feat_38)
    lines.append(f"- {flagged} of {len(feat_38)} focus features decoded most strongly to a format-like token.")
lines.append("- Feature edits below are relative to reconstruction, on a handful of Persian prompts.")
for row in sae_edits:
    lines.append(
        f"- Q{row['id']} ({row['group']}) feature {row['focus_feature']} token {row['feature_top_token']!r}: "
        f"{json.dumps(row['versus_reconstruction'])}"
    )
lines.append("")
lines.append("## Jacobian Lens")
lines.append("")
repeat = stability.get("repeat_fit")
lines.append(f"- Holdout readouts saved: {len(stability.get('holdout_readouts', []))}. Repeat-fit check: {repeat}.")
if readouts:
    key = str(SHARED_LAYER)
    vals = [row["content_token_jaccard"].get(key) for row in readouts if key in row["content_token_jaccard"]]
    if vals:
        lines.append(f"- Layer {SHARED_LAYER} mean content-token Jaccard: {float(np.mean(vals)):.3f}.")
lines.append("- Direction edits add an output embedding at the last residual. A changed letter is an effect of that edit.")
for row in lens_edits:
    lines.append(
        f"- Q{row['id']} ({row['group']}) content {row['en_content_token']!r} / {row['fa_content_token']!r}: "
        f"{json.dumps(row['edits'])}"
    )
lines.append("")
lines.append("## Activation patching")
lines.append("")
matched = [row for row in patches if row.get("control") == "matched"]
for direction in ("en_to_fa", "fa_to_en"):
    subset = [row for row in matched if row["direction"] == direction]
    if not subset:
        continue
    layer_38 = [row["layer_38_delta"] for row in subset if row["layer_38_delta"] is not None]
    lines.append(
        f"- {direction}: n={len(subset)}, mean Δmargin at layer {SHARED_LAYER} "
        f"{float(np.mean(layer_38)):.3f}, source-letter transfer "
        f"{float(np.mean([row['source_transferred'] for row in subset])):.2f}, "
        f"recovered {sum(bool(row['recovered']) for row in subset)}, "
        f"worsened {sum(bool(row['worsened']) for row in subset)}, "
        f"replacement ok {sum(bool(row['replacement_ok']) for row in subset)}/{len(subset)}."
    )
controls = [row for row in patches if row.get("control") != "matched"]
if controls:
    lines.append(
        "- Controls (mean |extreme Δmargin|): "
        + ", ".join(
            f"{name} {float(np.mean([abs(row['extreme_delta']) for row in controls if row['control'] == name])):.3f}"
            for name in ("same_language", "unrelated")
            if any(row["control"] == name for row in controls)
        )
        + "."
    )
gaps = [abs(row["source_tokens"] - row["target_tokens"]) for row in matched]
if gaps:
    lines.append(f"- Mean |source − target| token length on matched patches: {float(np.mean(gaps)):.1f}.")
lines.append("")
lines.append("## Layer 38 comparison")
lines.append("")
lines.append(f"- Rank correlations: {json.dumps(comparison.get('rank_correlations'))}")
lines.append(f"- {comparison.get('caveat', '')}")
lines.append("")
lines.append("## Case studies")
lines.append("")
feat_lookup = {(row["id"], row["layer"]): row for row in features}
lens_lookup = {row["id"]: row for row in readouts}
patch_lookup = {(row["id"], row["direction"]): row for row in matched}
for value in comparison.get("case_ids", []):
    base = by_id.get(int(value))
    if base is None:
        continue
    feat = feat_lookup.get((int(value), SHARED_LAYER), {})
    lens = lens_lookup.get(int(value), {})
    en_fa = patch_lookup.get((int(value), "en_to_fa"), {})
    fa_en = patch_lookup.get((int(value), "fa_to_en"), {})
    lines.append(f"### Question {value} ({base['group']})")
    lines.append("")
    lines.append(
        f"1. English predicted {base['languages']['english']['pred_letter']} "
        f"(correct={base['languages']['english']['correct']}, p={base['languages']['english']['p_correct']:.3f}); "
        f"Persian predicted {base['languages']['persian']['pred_letter']} "
        f"(correct={base['languages']['persian']['correct']}, p={base['languages']['persian']['p_correct']:.3f}). "
        f"Correct letter {base['correct_letter']}. JSD {base['jsd']['english_persian']:.3f}."
    )
    lines.append(
        f"2. SAE at layer {SHARED_LAYER}: top-feature Jaccard {feat.get('jaccard')}. "
        f"Largest activation gap is feature {feat.get('largest_diff_feature')}, "
        f"whose decoder top token is {feat.get('feature_top_token')!r} "
        f"(format-like={feat.get('format_like')})."
    )
    content = (lens.get("content_token_jaccard") or {}).get(str(SHARED_LAYER))
    lines.append(
        f"3. JLens content-token Jaccard at layer {SHARED_LAYER}: {content}. "
        "High-ranked tokens that are letters or template words are not treated as concepts."
    )
    lines.append(
        f"4. Patch EN→FA at layer {SHARED_LAYER}: Δmargin {en_fa.get('layer_38_delta')}, "
        f"letter {en_fa.get('target_pred')}→{en_fa.get('patched_pred')}, "
        f"source transferred={en_fa.get('source_transferred')}, recovered={en_fa.get('recovered')}, "
        f"worsened={en_fa.get('worsened')}. "
        f"FA→EN Δmargin at layer {SHARED_LAYER}: {fa_en.get('layer_38_delta')}."
    )
    lines.append(
        "5. The three results describe the same residual position. "
        "Use them together only where the behavioural group, the feature or token overlap, "
        "and the signed patch effect point the same way. Do not equate a shared layer with a shared concept."
    )
    lines.append("")
lines.append("## Six-criterion comparison")
lines.append("")
lines.append("- Internal representation: SAE exposes a sparse code of the residual. JLens exposes tokens after a linear map toward the final residual. Patching exposes the causal effect of replacing that residual, not a feature or a token.")
lines.append("- Important layers: SAE is reported at 20 and 38. JLens is reported at the fitted source layers, with 38 as the shared layer. Patching is summarised across 0–41, and layer 38 is the comparison point. The layer with the largest patch effect is not assumed to be the most interpretable.")
lines.append("- Interpretability: SAE features need examples and a check that they are not format or BOS directions. JLens tokens are readable and still mix answer letters with content. Patch curves are readable as signed effects and do not name a concept.")
lines.append("- Causal evidence: SAE ablation and steering are causal relative to reconstruction, on a few items. JLens direction edits are causal relative to the clean residual, on a few items. Patching is the systematic causal measurement, including both directions and the two controls.")
lines.append("- Computational requirements: see stage_times.json for seconds and peak GPU memory. Jacobian fitting dominates. Screening and SAE activation extraction are the other long stages.")
lines.append("- Multilingual suitability: all three can be applied to parallel prompts. SAE and JLens overlaps are descriptive correspondences across tokenisers. Patching is the method that tests whether one language’s residual changes the other language’s decision. Bengali and Urdu stay in the behavioural supplement.")
lines.append("")
lines.append("## Limitations")
lines.append("")
for item in (
    "Failed or skipped stages are listed in records/status.json.",
    "The mechanistic subset is small and disagreement-enriched.",
    "No autointerp labels are used as evidence of meaning.",
    "JLens stability across a repeated fit runs only when RUN_JLENS_REPEAT_FIT is True.",
    "Layer 29 is included only when RUN_JLENS_LAYER_29 is True.",
    "Same-language and unrelated patch controls cover a few items, not the full subset.",
):
    lines.append(f"- {item}")
lines.append("")

notes_path = REC / "report_notes.md"
notes_path.write_text("\n".join(lines) + "\n", encoding="utf-8")
print("saved", notes_path)
print("status", STATUS)


In [ ]:
# Checkpoints are only needed while later cells reload them.
# After analysis has written its JSON, drop the .pt files so output/ stays small.
groups = {
    "sae": (SAE_DIR, "sae_analysis"),
    "jlens": (JLENS_DIR, "jlens_analysis"),
}
removed = []
kept = []
for name, (directory, stage) in groups.items():
    paths = [path for path in directory.rglob("*.pt") if path.is_file()]
    if STATUS.get(stage) == "ok":
        removed.extend(paths)
    elif paths:
        kept.append(name)

freed = 0
for path in removed:
    freed += path.stat().st_size
    path.unlink()
print(f"removed {len(removed)} checkpoint files ({freed / 1e9:.2f} GB)")
if kept:
    print("kept checkpoints because analysis did not finish:", ", ".join(kept))
